# Day 4 — API Testing with Python

**Module 2 · Python for AI Testing & Automation**

---

## What we'll cover

| # | Topic | Why it matters |
|---|---|---|
| 1 | HTTP fundamentals | Every LLM is behind an HTTP API |
| 2 | `requests` — raw HTTP | The swiss-army knife for any HTTP call |
| 3 | `.env` and `python-dotenv` | Keep secrets out of code and Git |
| 4 | The OpenAI SDK | Cleaner than raw HTTP for OpenAI-compatible APIs |
| 5 | The Anthropic SDK | Different shape — different parsing |
| 6 | Multi-provider `LLMClient` | One interface, any backend |
| 7 | Response normalization | Standardize output regardless of provider |

**Prerequisite:** `.env` file with at least one valid key (`OPENAI_API_KEY` or Ollama running).

---

---
## 1. HTTP Fundamentals

Every LLM API call is an HTTP request. Before you use a SDK, you must understand the underlying protocol.

> **Plain English:** HTTP is sending a letter and receiving a reply. The **method** is the verb on the envelope (GET = "give me this", POST = "here's something new"). The **URL** is the address. The **headers** are the envelope markings — authentication lives here. The **body** is the letter itself. The **status code** is the postal stamp on the reply.

### Status codes you must know

| Code | Meaning | LLM context |
|---|---|---|
| `200` | OK | Request succeeded — response is in the body |
| `400` | Bad Request | Your payload was malformed |
| `401` | Unauthorized | API key missing or wrong |
| `403` | Forbidden | Valid key, no permission for this action |
| `422` | Unprocessable | Request shape is valid but parameter values are wrong |
| `429` | Too Many Requests | Rate limited — back off and retry |
| `500` | Server Error | Provider is having issues — retry later |
| `503` | Service Unavailable | Provider is down — retry later |

In [ ]:
# Install check
try:
    import requests
    print(f"requests {requests.__version__}")
except ImportError:
    print("Run: pip install requests")

---
## 2. `requests` — Raw HTTP

`requests` is Python's most popular HTTP library. It lets you make any HTTP call without dealing with raw sockets.

In [ ]:
import requests

# GET request — retrieve a resource
resp = requests.get("https://httpbin.org/get", timeout=10)

print(f"Status code : {resp.status_code}")
print(f"Content-Type: {resp.headers['Content-Type']}")
print(f"URL called  : {resp.json()['url']}")

In [ ]:
# POST request with headers and JSON body — this is the LLM API pattern
resp = requests.post(
    "https://httpbin.org/post",
    headers={
        "Authorization": "Bearer fake-api-key",
        "X-Custom-Header": "test",
    },
    json={   # automatically sets Content-Type: application/json
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": "Hello"}],
        "temperature": 0.3,
    },
    timeout=10,
)

print(f"Status: {resp.status_code}")
data = resp.json()
print(f"Echo of JSON body: {data['json']}")
print(f"Auth header seen: {data['headers'].get('Authorization', 'not sent')}")

In [ ]:
# Handling HTTP errors correctly
def safe_post(url: str, payload: dict, headers: dict) -> dict | None:
    """POST with proper error handling — returns parsed JSON or None."""
    try:
        resp = requests.post(url, json=payload, headers=headers, timeout=30)
        resp.raise_for_status()   # raises HTTPError for 4xx/5xx
        return resp.json()

    except requests.exceptions.Timeout:
        print("Request timed out after 30s")
    except requests.exceptions.ConnectionError as e:
        print(f"Network error: {e}")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP {e.response.status_code}: {e}")
        if e.response.status_code == 401:
            print("  → Check your API key")
        elif e.response.status_code == 429:
            print("  → Rate limited — add a delay and retry")
    except requests.exceptions.RequestException as e:
        print(f"Unexpected requests error: {e}")
    return None

# Test with a URL that returns a 404
result = safe_post("https://httpbin.org/status/404", {}, {})
print(f"Result: {result}")

In [ ]:
# Calling an Ollama endpoint with raw requests — no SDK needed
import os
OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434") + "/api/generate"

payload = {
    "model": "llama3.2:3b",
    "prompt": "What is 2 + 2? Answer with just the number.",
    "stream": False,
}

try:
    resp = requests.post(OLLAMA_URL, json=payload, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    print(f"Response: {data['response'].strip()}")
    print(f"Tokens  : {data.get('eval_count', 'N/A')}")
except requests.exceptions.ConnectionError:
    print("Ollama not running — start with: ollama serve")
except requests.exceptions.HTTPError as e:
    print(f"HTTP error: {e}")

---
## 3. `.env` and Secrets

**Rule #1:** API keys never go in source code. Never.

**Rule #2:** `.env` goes in `.gitignore`. Always.

`.env.example` (committed) shows what keys are needed with fake values. `.env` (gitignored) holds the real values.

> **Plain English:** an `.env` file is a sticky note you keep on your desk — with your actual passwords. You never photocopy it and mail it to people. `.env.example` is the blank template you hand to colleagues: "put your own values here."

In [ ]:
from dotenv import load_dotenv
import os

# Load .env file into os.environ
load_dotenv()   # reads .env from the current directory and parents

# Access keys — two patterns:
openai_key = os.environ.get("OPENAI_API_KEY")          # None if missing
provider   = os.getenv("PROVIDER", "ollama")            # default if missing
model      = os.getenv("DEMO_MODEL", "llama3.2:3b")

# Never print full secrets — mask them
if openai_key:
    masked = openai_key[:8] + "..." + openai_key[-4:]
    print(f"OpenAI key loaded: {masked}")
else:
    print("OPENAI_API_KEY not set — using Ollama")

print(f"Provider : {provider}")
print(f"Model    : {model}")

In [ ]:
# Fail-fast if a required key is missing
def require_env(key: str) -> str:
    """Return env var value or raise a clear error — fail fast, fail clearly."""
    value = os.environ.get(key)
    if not value:
        raise EnvironmentError(
            f"Required environment variable {key!r} is not set.\n"
            f"Add it to your .env file or export it before running."
        )
    return value

# Only run this if you have an OpenAI key — otherwise skip
if os.environ.get("OPENAI_API_KEY"):
    key = require_env("OPENAI_API_KEY")
    print(f"Got key: {key[:8]}...")
else:
    print("Skipping require_env demo (no OPENAI_API_KEY set)")

---
## 4. The OpenAI SDK

The `openai` Python package wraps the raw HTTP API. It handles:
- Authentication
- Request serialization/deserialization
- Retries on transient errors
- Streaming

Because Ollama and other providers expose an OpenAI-compatible API, this one client covers multiple backends.

In [ ]:
from openai import OpenAI
import os

# Initialize for Ollama (default)
if os.getenv("PROVIDER", "ollama") == "openai":
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
else:
    client = OpenAI(
        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        api_key="ollama",   # Ollama doesn't check the key, any string works
    )

print(f"Client created for: {os.getenv('PROVIDER', 'ollama')}")

In [ ]:
MODEL = os.getenv("DEMO_MODEL", "llama3.2:3b")

# Basic chat completion
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer in one sentence."},
        {"role": "user",   "content": "What is a neural network?"},
    ],
    temperature=0.3,
    max_tokens=100,
)

print("=== Response Object ===")
print(f"Model          : {resp.model}")
print(f"Finish reason  : {resp.choices[0].finish_reason}")
print(f"Content        : {resp.choices[0].message.content}")
print(f"Prompt tokens  : {resp.usage.prompt_tokens}")
print(f"Completion tkns: {resp.usage.completion_tokens}")
print(f"Total tokens   : {resp.usage.total_tokens}")

In [ ]:
# Multi-turn conversation — build message history
conversation = [
    {"role": "system", "content": "You are a concise AI testing expert."},
]

questions = [
    "What is hallucination in an LLM?",
    "How do you test for it?",
    "What tool would you use in production?",
]

for q in questions:
    conversation.append({"role": "user", "content": q})
    resp = client.chat.completions.create(
        model=MODEL,
        messages=conversation,
        temperature=0.3,
        max_tokens=120,
    )
    answer = resp.choices[0].message.content.strip()
    conversation.append({"role": "assistant", "content": answer})
    print(f"Q: {q}")
    print(f"A: {answer}\n")

print(f"Total conversation turns: {len(conversation)}")

In [ ]:
# Streaming — receive tokens as they're generated
print("Streaming response (each token as it arrives):")
print("-" * 50)

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 10, one number per line."}],
    stream=True,
    max_tokens=80,
)

full_response = ""
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        full_response += delta

print("\n" + "-" * 50)
print(f"Total chars received: {len(full_response)}")

---
## 5. The Anthropic SDK

The Anthropic SDK has a different API shape. Understanding both prepares you for the multi-provider client.

In [ ]:
# Only run this if you have an Anthropic API key
import os
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY not set — skipping Anthropic demo")
    print("To run: add ANTHROPIC_API_KEY=sk-ant-... to your .env file")
else:
    import anthropic
    anth_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    # Key differences from OpenAI:
    # 1. system prompt is a separate top-level parameter, NOT in messages
    # 2. max_tokens is required (not optional)
    # 3. response.content is a LIST of blocks, not a single string
    resp = anth_client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=100,
        system="You are a concise assistant. Answer in one sentence.",
        messages=[
            {"role": "user", "content": "What is the context window of an LLM?"}
        ],
    )

    # Navigate the response shape
    print(f"Model        : {resp.model}")
    print(f"Stop reason  : {resp.stop_reason}")
    print(f"Content type : {resp.content[0].type}")
    print(f"Answer       : {resp.content[0].text}")
    print(f"Input tokens : {resp.usage.input_tokens}")
    print(f"Output tokens: {resp.usage.output_tokens}")

---
## 6. Multi-Provider `LLMClient` — The Unified Interface

The goal: one `generate()` function that works with any provider. Callers don't know or care which backend is behind it.

> **Design principle:** hide provider-specific complexity behind a stable interface. If you switch from OpenAI to Claude, you change one environment variable — not 50 call sites.

In [ ]:
from dataclasses import dataclass
from typing import Optional
import time

@dataclass
class LLMResponse:
    """Normalized response from any LLM provider."""
    content:       str
    model:         str
    provider:      str
    prompt_tokens: int
    output_tokens: int
    latency_ms:    float
    finish_reason: str = "stop"

    @property
    def total_tokens(self) -> int:
        return self.prompt_tokens + self.output_tokens

    def __str__(self) -> str:
        return (
            f"[{self.provider}/{self.model}] "
            f"{self.latency_ms:.0f}ms | "
            f"{self.total_tokens} tokens\n"
            f"{self.content}"
        )

print("LLMResponse dataclass defined")

In [ ]:
class LLMClient:
    """
    Multi-provider LLM client.
    Provider is selected by the PROVIDER env var: 'ollama', 'openai', 'anthropic'.
    Returns a normalized LLMResponse regardless of provider.
    """

    def __init__(
        self,
        provider: str | None = None,
        model: str | None = None,
        temperature: float = 0.3,
        max_tokens: int = 500,
    ):
        self.provider    = (provider or os.getenv("PROVIDER", "ollama")).lower()
        self.model       = model or os.getenv("DEMO_MODEL", "llama3.2:3b")
        self.temperature = temperature
        self.max_tokens  = max_tokens

    def generate(self, prompt: str, system: str = "") -> LLMResponse:
        """Generate a response. Raises on unrecoverable errors."""
        if self.provider in ("openai", "ollama"):
            return self._call_openai_compat(prompt, system)
        if self.provider == "anthropic":
            return self._call_anthropic(prompt, system)
        raise ValueError(f"Unknown provider: {self.provider!r}")

    def _call_openai_compat(self, prompt: str, system: str) -> LLMResponse:
        from openai import OpenAI
        if self.provider == "openai":
            c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        else:
            c = OpenAI(
                base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
                api_key="ollama",
            )

        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        start = time.time()
        resp  = c.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
        )
        return LLMResponse(
            content       = resp.choices[0].message.content.strip(),
            model         = resp.model,
            provider      = self.provider,
            prompt_tokens = resp.usage.prompt_tokens,
            output_tokens = resp.usage.completion_tokens,
            latency_ms    = (time.time() - start) * 1000,
            finish_reason = resp.choices[0].finish_reason or "stop",
        )

    def _call_anthropic(self, prompt: str, system: str) -> LLMResponse:
        import anthropic
        c = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        start = time.time()
        kwargs = dict(
            model=self.model,
            max_tokens=self.max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        if system:
            kwargs["system"] = system
        resp = c.messages.create(**kwargs)
        return LLMResponse(
            content       = resp.content[0].text,
            model         = resp.model,
            provider      = "anthropic",
            prompt_tokens = resp.usage.input_tokens,
            output_tokens = resp.usage.output_tokens,
            latency_ms    = (time.time() - start) * 1000,
            finish_reason = resp.stop_reason or "stop",
        )

print("LLMClient defined")

In [ ]:
# Use the client
client = LLMClient()   # uses PROVIDER and DEMO_MODEL from env

prompts = [
    ("What is the transformer architecture?", ""),
    ("What is 15 × 7?", "Answer with just the number."),
    ("Name three LLM evaluation tools.", "List each on its own line."),
]

for prompt, system in prompts:
    resp = client.generate(prompt, system=system)
    print(resp)
    print("-" * 60)

---
## 7. Response Normalization & Validation

Now that we have a normalized response, we can write provider-agnostic assertions.

In [ ]:
def assert_response(
    resp: LLMResponse,
    min_length: int = 1,
    max_length: int = 2000,
    must_include: list[str] | None = None,
    must_not_include: list[str] | None = None,
    max_latency_ms: float = 10_000,
) -> tuple[bool, list[str]]:
    """Validate an LLMResponse against a set of behavioral constraints."""
    issues = []

    if len(resp.content) < min_length:
        issues.append(f"too short: {len(resp.content)} < {min_length} chars")
    if len(resp.content) > max_length:
        issues.append(f"too long: {len(resp.content)} > {max_length} chars")
    if resp.latency_ms > max_latency_ms:
        issues.append(f"too slow: {resp.latency_ms:.0f}ms > {max_latency_ms}ms")

    content_lower = resp.content.lower()
    for kw in (must_include or []):
        if kw.lower() not in content_lower:
            issues.append(f"missing required keyword: {kw!r}")
    for kw in (must_not_include or []):
        if kw.lower() in content_lower:
            issues.append(f"contains forbidden keyword: {kw!r}")

    return len(issues) == 0, issues


# Run validation on the previous responses
test_spec = {
    "prompt": "What is the transformer architecture?",
    "must_include": ["attention"],
    "must_not_include": ["LSTM"],
    "min_length": 50,
    "max_length": 1000,
}

client = LLMClient()
resp   = client.generate(test_spec["prompt"])
passed, issues = assert_response(
    resp,
    min_length       = test_spec["min_length"],
    max_length       = test_spec["max_length"],
    must_include     = test_spec["must_include"],
    must_not_include = test_spec["must_not_include"],
)

print(f"Response: {resp.content[:100]}...")
print(f"Result  : {'PASS' if passed else 'FAIL'}")
for issue in issues:
    print(f"  ✗ {issue}")

In [ ]:
# 🔧 Try it:
# 1. Create an LLMClient for a different provider (if you have keys)
# 2. Run the same prompt on two providers using a loop
# 3. Compare: latency_ms, total_tokens, content length
# 4. Which provider is faster? Which is more verbose?

# providers_to_test = ["ollama"]  # add "openai" or "anthropic" if you have keys
# for provider in providers_to_test:
#     c = LLMClient(provider=provider)
#     r = c.generate("Explain what a context window is. One paragraph.")
#     print(f"{provider}: {r.latency_ms:.0f}ms | {r.total_tokens} tokens")
#     print(r.content[:100])
#     print()

---
## Day 4 Summary

| Topic | Key API | Coming back in |
|---|---|---|
| HTTP fundamentals | Status codes, headers, body | Every API test |
| `requests` | `.post(url, json=..., headers=..., timeout=...)` | Module 3, Module 8 |
| `.env` | `load_dotenv()`, `os.getenv(key, default)` | Every module |
| OpenAI SDK | `client.chat.completions.create(...)` | Day 5, 6, 7 |
| Anthropic SDK | `client.messages.create(system=..., messages=...)` | Optional |  
| `LLMClient` | `client.generate(prompt)` → `LLMResponse` | Day 5, 6 (reuse this) |
| `assert_response` | Behavioral constraints on normalized response | Day 5 (pytest version) |

**Exercise:** [`exercises/day4_exercise.md`](../exercises/day4_exercise.md)  
**Next:** Day 5 — pytest: turning our manual checks into a proper test suite
